In [29]:
from Training_utils import train_loader, train
import torch_geometric as tg
import torch
from SUPG_prediction_models import *

from Training_utils import *


tset = graph_dataset(f"data/training_set_globalizer_wedge/input_values")


class gat(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = tg.nn.models.GAT(
            in_channels=12, 
            hidden_channels=5, 
            num_layers=10, 
            out_channels=1, 
            v2=True, 
            #dropout=0., 
            act=torch.relu, 
            #norm=torch_geometric.nn.norm.LayerNorm(1),
            add_self_loops=False,
            residual=False
        )
        self.a = torch.tensor([0.1], dtype=torch.float32)
        self.b = torch.tensor([0.1], dtype=torch.float32)

    def forward(self, data) -> torch.Tensor:
        x, edge_index, delta_uh, D_uh = data.x, data.edge_index, data.delta_uh, data.D_uh
        h = self.model(
            x=x,
            edge_index=edge_index
        )
        upper = torch.sigmoid(self.a)*delta_uh + torch.sigmoid(self.b)*D_uh
        return upper*torch.sigmoid(h)
    
batch_size = 15
loader = train_loader(batch_size=batch_size, set=tset)
model=gat()


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, factor=0.8, patience=50)
curr_loss = 10



In [32]:
batch_size = 5
loader = train_loader(batch_size=batch_size, set=tset)

In [33]:

for i in range(5000):
    loss = train(model=model, loader=loader, optimizer=optimizer, device='cpu')
    if curr_loss > loss:    
        print(f"iteration {i}: new loss: {loss}")
        curr_loss = loss
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss': loss}, "data/models/GATv2_supervised_globalizer_wedge.pth")
        #scheduler.step(loss)
    else:
        print(f"iteration {i}: {loss}")
        scheduler.step(loss)

        


iteration 0: 48.482795079549156
iteration 1: 47.54708735148112
iteration 2: 46.6339177025689
iteration 3: 45.74400287204318
iteration 4: 44.87677192687988
iteration 5: 44.03154542711046
iteration 6: 43.20759179857042
iteration 7: 42.404249827067055
iteration 8: 41.62085300021701
iteration 9: 40.85674900478787
iteration 10: 40.11140325334337
iteration 11: 39.38421090443929
iteration 12: 38.67456351386176
iteration 13: 37.98197725084093
iteration 14: 37.30588245391846
iteration 15: 36.64582432640923
iteration 16: 36.00130812327067
iteration 17: 35.371804767184784
iteration 18: 34.7569178475274
iteration 19: 34.1562106874254
iteration 20: 33.569310400221084
iteration 21: 32.9958061642117
iteration 22: 32.435287369622124
iteration 23: 31.88737466600206
iteration 24: 31.351686265733505
iteration 25: 30.827924092610676
iteration 26: 30.315723101298016
iteration 27: 29.814764234754776
iteration 28: 29.32473701900906
iteration 29: 28.845328754848904
iteration 30: 28.376238293117947
iteration 3

KeyboardInterrupt: 